<a href="https://colab.research.google.com/github/jimenezjos/Aplicaciones/blob/main/FlechayCojinete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Solución Analítica
Vamosa definir las variables aleatorias para el diámetro interno del cojinete ($X_1$) y el diámetro externo de la flecha ($X_2$).

Según los datos:

$X_1 \sim N(\mu_1 = 1.5, \sigma_1^2 = 0.0016)$$X_2 \sim N(\mu_2 = 1.48, \sigma_2^2 = 0.0009)$

**a) Probabilidad de que haya interferencia**

La interferencia ocurre cuando el diámetro de la flecha es mayor que el del cojinete, es decir, cuando $X_2 > X_1$. Para resolver esto, definimos una nueva variable aleatoria $Y$ que represente la holgura entre ambas piezas:$$Y = X_1 - X_2$$Habrá interferencia cuando $Y < 0$. Como $X_1$ y $X_2$ son distribuciones normales e independientes, la variable $Y$ también sigue una distribución normal con los siguientes parámetros:

Media de $Y$ ($\mu_Y$):

$$\mu_Y = \mu_1 - \mu_2 = 1.5 - 1.48 = 0.02$$

Varianza de $Y$ ($\sigma_Y^2$):(La varianza de una resta de variables independientes es la suma de sus varianzas)

$$\sigma_Y^2 = \sigma_1^2 + \sigma_2^2 = 0.0016 + 0.0009 = 0.0025$$Desviación estándar de $Y$ ($\sigma_Y$):$$\sigma_Y = \sqrt{0.0025} = 0.05$$Ahora calculamos la probabilidad de que $Y < 0$ estandarizando a la variable $Z$:$$P(Y < 0) = P\left(Z < \frac{0 - \mu_Y}{\sigma_Y}\right)$$$$P(Y < 0) = P\left(Z < \frac{0 - 0.02}{0.05}\right) = P(Z < -0.4)$$Buscando en las tablas de la distribución normal estándar (o usando software), encontramos que:La probabilidad de interferencia es $\approx 0.3446$ (o 34.46%).

**b) Número de veces necesario para simular el experimento**

Se nos pide calcular el tamaño de muestra $n$ para estimar una proporción (la probabilidad de interferencia $p$) con un margen de error máximo $E = 0.01$ y un nivel de confianza del 95%.
La fórmula para el tamaño de muestra de una proporción es:$$n = \frac{Z_{\alpha/2}^2 \cdot p(1-p)}{E^2}$$Donde:$Z_{\alpha/2} \approx 1.96$ (valor $Z$ para un 95% de confianza).$p = 0.3446$ (la verdadera probabilidad calculada en el inciso anterior).$E = 0.01$.Sustituyendo los valores:$$n = \frac{(1.96)^2 \cdot (0.3446)(1 - 0.3446)}{(0.01)^2}$$$$n = \frac{3.8416 \cdot (0.3446)(0.6554)}{0.0001}$$$$n = \frac{3.8416 \cdot 0.22585}{0.0001} \approx 8676.25$$Para asegurar el margen de error, redondeamos al entero superior.Se requiere simular el experimento al menos 8,677 veces.

In [4]:
import math
import random
import scipy.stats as stats
def generar_normal_box_muller(mu, sigma, n_simulaciones):
    resultados = []

    while len(resultados) < n_simulaciones:
        # 1. Generar U1, U2 ~ U(0,1)
        u1 = random.random()
        u2 = random.random()

        while u1 == 0:
            u1 = random.random()

        raiz = math.sqrt(-2.0 * math.log(u1))
        angulo = 2.0 * math.pi * u2

        z1 = raiz * math.cos(angulo)
        z2 = raiz * math.sin(angulo)

        resultados.append(mu + (z1 * sigma))
        if len(resultados) < n_simulaciones:
            resultados.append(mu + (z2 * sigma))

    return resultados

mu_cojinete = 1.5
sigma_cojinete = math.sqrt(0.0016)
mu_flecha = 1.48
sigma_flecha = math.sqrt(0.0009)

n_simulaciones = 8677

cojinetes = generar_normal_box_muller(mu_cojinete, sigma_cojinete, n_simulaciones)
flechas = generar_normal_box_muller(mu_flecha, sigma_flecha, n_simulaciones)

interferencias = sum(1 for c, f in zip(cojinetes, flechas) if f > c)

probabilidad_simulada = interferencias / n_simulaciones

print(f"Probabilidad de interferencia simulada con Box-Muller: {probabilidad_simulada:.4f}")

nivel_confianza = 0.95
error_maximo = 0.01

# Calcular el valor Z para un 95% de confianza (dos colas)
# Para 95%, alfa = 0.05. Cada cola tiene 0.025.
# stats.norm.ppf es la función inversa; nos da el valor Z exacto de la tabla.
alfa = 1 - nivel_confianza
z_alfa_medios = abs(stats.norm.ppf(alfa / 2))

# Aplicar la fórmula del tamaño de muestra para una proporción
n_calculado = (z_alfa_medios**2 * probabilidad_simulada * (1 - probabilidad_simulada)) / (error_maximo**2)

# El tamaño de muestra SIEMPRE se debe redondear al entero superior (techo)
# para garantizar que el error sea estrictamente menor a 0.01.
n_final = math.ceil(n_calculado)

print("-" * 50)
print(f"b) Nivel de confianza: {nivel_confianza*100}%")
print(f"b) Valor Z exacto calculado: {z_alfa_medios:.4f}")
print(f"b) Tamaño de muestra crudo: {n_calculado:.2f}")
print(f"b) RESULTADO: Número de simulaciones a realizar: {n_final}")

Probabilidad de interferencia simulada con Box-Muller: 0.3430
--------------------------------------------------
b) Nivel de confianza: 95.0%
b) Valor Z exacto calculado: 1.9600
b) Tamaño de muestra crudo: 8656.47
b) RESULTADO: Número de simulaciones a realizar: 8657
